# Seminar 08: PixelCNN for Image Modeling

This notebook demonstrates a simplified implementation of PixelCNN.

The notebook covers:
1. Data preprocessing: bucketizing image pixel values from 0-255 into 10 discrete bins.
2. Defining image transformations including resizing and converting images to grayscale.
3. Creating a mask for masked convolution layers, crucial for the autoregressive property.
4. Building a simple PixelCNN model using masked convolutions.
5. Training and autoregressive sample generation.

Date: 2025-03-04

In [ ]:
import os
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torchvision.transforms as transforms

import matplotlib.pyplot as plt

from IPython.display import clear_output

# Set device
cuda = torch.cuda.is_available()
device = torch.device("cuda" if cuda else ("mps" if torch.backends.mps.is_available() else "cpu"))
torch.backends.cudnn.benchmark = cuda

print(f"Using device: {device}")

In [ ]:
# Data Parameters
original_shape = [268, 182]  # Original dimensions of the poster images
resize_n = 3  # Factor by which the image will be downscaled

# Loader Parameters
batch_size = 32
num_workers = 4 * cuda
pin_memory = cuda

## Task 1: Bucketize Pixel Values

Write a function that takes as input a tensor of shape
(batch x channels x width x height), where each element is an integer between 0 and 255,
and returns a tensor of the same shape but with each element transformed to an integer between 0 and 9.


In [ ]:
def bucketize(x):
    """
    Converts a tensor with integer values in the range [0, 255] to a tensor with values in the range [0, 9].
    
    We divide the range 0-255 into 10 equal buckets. Using a division factor of 26 
    (since 10*26 = 260) and clamping the maximum value to 9 ensures that values near 255
    are assigned to bucket 9.
    
    Args:
        x (torch.Tensor): Input tensor with values in the range [0, 255].
    
    Returns:
        torch.Tensor: Tensor with bucketized values in the range [0, 9].
    """
    # TODO: YOUR CODE

## Task 2: Image Transformations

Each image is a PIL image in the format `3 x original_shape[0] x original_shape[1]`.
The transformation pipeline needs to:
1. Resize the image by a factor of `resize_n`.
2. Convert the image to grayscale (i.e., a black-and-white image).
3. Convert the PIL image to a tensor.
4. Apply the bucketize function.

In [ ]:
poster_transforms = transforms.Compose([
    # Resize the image to (original height // resize_n, original width // resize_n)
    # TODO: YOUR CODE
    # Convert the image to grayscale (single channel)
    # TODO: YOUR CODE
    # Convert the PIL image to a tensor with values in [0, 1]
    transforms.ToTensor(),
    # Multiply by 255 to bring values back to [0, 255] and then bucketize
    transforms.Lambda(lambda x: bucketize(x * 255))
])

### Load Data

In [ ]:
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image, UnidentifiedImageError

class CustomImageDataset(Dataset):
    """
    A custom dataset for loading images from a single folder.
    It verifies each image file during initialization and filters out invalid ones.
    """
    def __init__(self, root, transform=None):
        """
        Args:
            root (str): Directory with all the images.
            transform (callable, optional): Optional transform to be applied on an image.
        """
        self.root = root
        self.transform = transform
        # List image files (check common image extensions)
        candidate_files = sorted([
            os.path.join(root, f) for f in os.listdir(root)
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))
        ])
        self.image_files = []
        for f in candidate_files:
            try:
                # Verify if file is an image
                with Image.open(f) as img:
                    img.verify()  # Verify the file without decoding the image data
                self.image_files.append(f)
            except (UnidentifiedImageError, IOError) as e:
                print(f"Skipping file {f} due to error: {e}")
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        # Open image and ensure it's in RGB format
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        # Returning a dummy label (0) for compatibility with DataLoader.
        return image, 0

In [ ]:
# Path to the dataset directory
data_path = "../data/movie-genre-from-its-poster/posters/"

# Create the custom dataset
dataset = CustomImageDataset(data_path, transform=poster_transforms)

# Total number of images
num_total = len(dataset)
print(f"Total images found: {num_total}")

In [ ]:
# Define indices for splitting:
# - Last 100 images as validation (based on the natural file order)
# - The rest as training
train_indices = list(range(0, num_total - 100))
val_indices = list(range(num_total - 100, num_total))


# Create Subset datasets for training and validation
train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices)
print(f"Training images: {len(train_dataset)}, Validation images: {len(val_dataset)}")

In [ ]:
train_loader = DataLoader(dataset=train_dataset,
                          batch_size=batch_size,
                          shuffle=True,
                          drop_last=True,
                          num_workers=num_workers,
                          pin_memory=pin_memory)

val_loader = DataLoader(dataset=val_dataset,
                        batch_size=batch_size,
                        shuffle=True,
                        drop_last=True,
                        num_workers=num_workers,
                        pin_memory=pin_memory)

In [ ]:
image, _ = next(iter(train_loader))
print("Image batch shape:", image.shape)

In [ ]:
def plot_image(image, cmap=None):
    """
    Plots an image tensor.
    
    Args:
        image (torch.Tensor): Image tensor with shape [channels, height, width].
        cmap (str): Optional color map for display.
    """
    plt.figure(figsize=(10, 10))
    plt.imshow(image.detach().cpu().permute(1, 2, 0).squeeze(), cmap=cmap)
    plt.axis('off')
    plt.show()


## PixelCNN Overview

Recall that PixelCNN is an autoregressive model which models the joint distribution of an image as:

$$
p(x) = \prod_{i=1}^{N} p(x_i \mid x_1, \ldots, x_{i-1}),
$$

where each pixel is conditioned on all previous pixels in a specified ordering.

Instead of using recurrent networks (like PixelRNN), PixelCNN uses masked convolutional layers
to ensure that the prediction for each pixel only depends on already generated pixels.

Below is an illustration of the masked convolution concept:

<p align="center">
    <img src="https://raw.githubusercontent.com/aleju/papers/master/neural-nets/images/Conditional_Image_Generation_with_PixelCNN_Decoders__masked_convolution.png" />
</p>

## Task 3: Generate the Convolution Mask

Write a function that generates a mask for a convolution kernel.

The mask is created so that:
- All pixels below the center (in later rows) are masked out.
- In the center row, all pixels to the right of the center are masked out.
- Optionally, the center pixel can be masked out (if `include_center` is False).

In [ ]:
def make_mask(include_center, height, width):
    """
    Generates a mask for a convolution kernel used in PixelCNN.
    
    For a kernel of size (height, width):
        - Pixels in rows after the center are set to 0.
        - In the center row, pixels to the right of the center are set to 0.
        - If include_center is False, the center pixel is also set to 0.
    
    Args:
        include_center (bool): Whether to include the center pixel.
        height (int): Height of the convolution kernel.
        width (int): Width of the convolution kernel.
    
    Returns:
        torch.Tensor: A mask tensor of shape (height, width) with values 0 or 1.
    """
    # TODO: TOUR CODE

## Masked Convolution Layer

Below is the implementation of a masked convolution layer, `MaskedCNN`, that applies
the mask generated by `make_mask` to the convolutional weights.

In [ ]:
class MaskedCNN(nn.Conv2d):
    """
    A 2D convolution layer with a mask applied to its weights.
    
    The mask is generated using the `make_mask` function and is applied to every kernel,
    ensuring that the network adheres to the autoregressive property.
    """
    def __init__(self, include_center, *args, **kwargs):
        super(MaskedCNN, self).__init__(*args, **kwargs)
        self.include_center = include_center
        # Get the kernel dimensions
        _, _, height, width = self.weight.size()
        # Create the 2D mask and expand it to match the weight dimensions:
        # [out_channels, in_channels, height, width]
        mask_2d = make_mask(self.include_center, height, width)
        mask_full = mask_2d.unsqueeze(0).unsqueeze(0).expand(self.weight.size(0), self.weight.size(1), height, width)
        self.register_buffer('mask', mask_full)
        
    def forward(self, x):
        # Apply the mask to the weights before the convolution
        self.weight.data *= self.mask
        return super(MaskedCNN, self).forward(x)

## Task 4: Define the PixelCNN Model

Below we define a simple PixelCNN model for grayscale images.
The network consists of:
- A masked convolution layer of type A (excluding the center pixel).
- A masked convolution layer of type B (including the center pixel).
- A final 1x1 convolution that outputs logits for each pixel (10 buckets).

Additionally, the model includes a `generate_samples` method to perform autoregressive
image generation by iteratively sampling pixel values.

In [ ]:
class PixelCNN(nn.Module):
    """
    A simple implementation of the PixelCNN model for grayscale images.
    
    The model uses masked convolution layers to ensure the autoregressive property.
    The final output consists of logits for each of the 10 pixel buckets.
    """
    def __init__(self, input_channels=1, num_pixels=10):
        super(PixelCNN, self).__init__()
        self.num_pixels = num_pixels
        self.layers = nn.Sequential(
            # TODO: YOUR CODE
        )
        
    def forward(self, x):
        """
        Forward pass of the PixelCNN model.
        
        Args:
            x (torch.Tensor): Input tensor of shape [batch, channels, height, width].
            
        Returns:
            torch.Tensor: Logits of shape [batch, num_pixels, height, width].
        """
        x = x.float()
        x = self.layers(x)
        return x
        
    def generate_samples(self, starting_image, starting_point=(0, 0)):
        """
        Autoregressively generates samples for the missing pixels in the image.
        
        Args:
            starting_image (torch.Tensor): Partially filled image tensor of shape [batch, channels, height, width].
            starting_point (tuple): Coordinates (row, col) where sampling begins.
        
        Returns:
            torch.Tensor: The completed image tensor.
        """
        self.eval()
        # TODO: YOUR CODE

In [ ]:
def plot_losses(losses):
    """
    Plots the training losses.
    
    Args:
        losses (list): List of loss values.
    """
    plt.figure(figsize=(10, 4))
    plt.plot(losses)
    plt.xlabel("Iteration")
    plt.ylabel("Loss")
    plt.title("Training Loss")
    plt.show()

In [ ]:
# Instantiate the PixelCNN model, optimizer, and loss criterion.
model = PixelCNN().to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

# Test a forward pass on a sample batch
logits = model(image.to(device))
print("Logits shape:", logits.shape)  # Expected shape: [batch, 10, height, width]

In [ ]:
# Training loop with sample generation demonstration
losses = []
starting_x, starting_y = 10, 10  # Starting coordinates for generation
n_epoch = 50

for epoch in range(n_epoch):
    model.train()
    for ind, (image, _) in tqdm(enumerate(train_loader), total=len(train_loader), leave=False):
        optimizer.zero_grad()
        image = image.to(device)
        logits = model(image)
        # The target should have shape [batch, height, width] (squeeze the channel dimension)
        loss = criterion(logits, image.long().squeeze(1))
        loss.backward()
        losses.append(loss.item())
        optimizer.step()
    clear_output(wait=True)
    model.eval()
    image, _ = next(iter(val_loader))
    # Create a starting image by zeroing out pixels in the bottom-right region
    starting_image = image.clone()
    starting_image[:, :, starting_x:, starting_y:] = 0
    sample = model.generate_samples(starting_image[:2].to(device), (starting_x, starting_y))
    plot_image(sample[0], cmap='gray')
    plot_losses(losses)

In [ ]:
# Final evaluation: Generate samples on a batch with a different masked region.
model.eval()
starting_image = image[:8].clone()
starting_image[:, :, 30:, 30:] = 0
sample = model.generate_samples(starting_image.to(device), starting_point=(30, 30))

for i in sample:
    plot_image(i, cmap='gray')

## Final Question

**Question:** What should we do when our images are not grayscale, but color images?
